# Fashion Retrieval — Full Evaluation (All Configs)

Runs end-to-end retrieval evaluation for **Config A, B, and C** on the entire query set.

---

### Metrics reported
| Metric | Description |
|--------|-------------|
| Recall@K | 1 if at least one correct item appears in top-K |
| NDCG@K | Discounted cumulative gain — rewards higher-ranked correct items |
| mAP@K | Mean average precision — penalises correct items appearing late |

Reported at K ∈ {5, 10, 15} as mean ± std across team roll number seeds.

---

### Ground truth
A retrieved gallery item is a correct match if and only if its `item_id` equals the query's `item_id`.

---

### Inputs required
- `vr-yolo-bbox-cropped-images` — query crops + master_crops.csv
- clip indexes AB dataset — FAISS indexes A and B + gallery_metadata.csv
- clip finetuned indexes C dataset — FAISS indexes C + clip_finetuned_full.pt

## 1. Install Packages

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.8 MB/s eta 0:00:00


## 2. Configuration

**Replace `ROLL_SEEDS` with your actual team roll numbers before running.**

In [2]:
"""
Fashion Retrieval — Batch Evaluation
Covers all ablation configs: A, B (beta=0.7, 0.5), C (beta=0.7, 0.5)
Metrics: Recall@K, NDCG@K, mAP@K  for K in {5, 10, 15}
Aggregation: mean ± std over multiple random seeds
"""

import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

# ── Dataset paths ─────────────────────────────────────────────────────────────
CROP_DATA_DIR    = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
AB_INDEX_DIR     = '/kaggle/input/datasets/akibatra25/clip-ab-output'
C_INDEX_DIR      = '/kaggle/input/datasets/akibatra25/clip-c-output'

# ── Seeds — use your roll numbers ─────────────────────────────────────────────
# ============================================================
ROLL_SEEDS = [25, 29, 513, 521]  
# ============================================================

# ── Evaluation settings ───────────────────────────────────────────────────────
K_LIST      = [5, 10, 15]
ENC_BATCH   = 64
CLIP_CKPT   = 'openai/clip-vit-base-patch32'

# ── Runtime device ────────────────────────────────────────────────────────────
GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device      : {GPU}')
print(f'Roll seeds  : {ROLL_SEEDS}')
print(f'K values    : {K_LIST}')
print()

for tag, p in [
    ('CROP_DATA_DIR', CROP_DATA_DIR),
    ('AB_INDEX_DIR',  AB_INDEX_DIR),
    ('C_INDEX_DIR',   C_INDEX_DIR),
]:
    ok = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{ok}] {tag}: {p}')

Device      : cuda
Roll seeds  : [25, 29, 513, 521]
K values    : [5, 10, 15]

[Found ✓] CROP_DATA_DIR: /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images
[Found ✓] AB_INDEX_DIR: /kaggle/input/datasets/akibatra25/clip-ab-output
[Found ✓] C_INDEX_DIR: /kaggle/input/datasets/akibatra25/clip-c-output


## 3. Load Query Images and Gallery Metadata

In [3]:
print('Loading CSVs...')

full_table = pd.read_csv(os.path.join(CROP_DATA_DIR, 'master_crops.csv'))
qry_table  = full_table[full_table['split'] == 'query'].reset_index(drop=True)

def translate_path(stored):
    if pd.isna(stored):
        return stored
    tail = stored.replace('/kaggle/working/', '').replace('/kaggle/input/', '')
    for ds in ['vr-yolo-bbox-cropped-images/',
               'datasets/akibatra25/vr-yolo-bbox-cropped-images/']:
        tail = tail.replace(ds, '')
    return os.path.join(CROP_DATA_DIR, tail)

qry_table['img_path'] = qry_table['crop_path'].apply(translate_path)
qry_table['on_disk']  = qry_table['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

# Fallback: construct paths directly if remapping gave poor coverage
if qry_table['on_disk'].sum() < len(qry_table) * 0.9:
    print('Trying direct path construction...')
    def direct_path(name):
        rel = name[4:] if name.startswith('img/') else name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(CROP_DATA_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(CROP_DATA_DIR, 'data/bbox_crops', rel)
    qry_table['img_path'] = qry_table['image_name'].apply(direct_path)
    qry_table['on_disk']  = qry_table['img_path'].apply(os.path.exists)

valid_qry = qry_table[qry_table['on_disk']].reset_index(drop=True)
print(f'Query images found : {len(valid_qry):,} / {len(qry_table):,}')

# Gallery metadata — maps FAISS index positions to item_ids
gal_meta   = pd.read_csv(os.path.join(AB_INDEX_DIR, 'item_index_map.csv'))
gal_ids    = gal_meta['item_id'].tolist()
gal_counts = gal_meta['item_id'].value_counts().to_dict()

print(f'Gallery rows       : {len(gal_meta):,}')
print(f'Unique gallery ids : {gal_meta["item_id"].nunique():,}')

Loading CSVs...
Query images found : 14,218 / 14,218
Gallery rows       : 12,612
Unique gallery ids : 3,985


## 4. Load CLIP and Encode Query Images

In [4]:
print(f'Loading CLIP: {CLIP_CKPT}')
clip_proc = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_net  = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)
for p in clip_net.parameters():
    p.requires_grad = False
clip_net.eval()
VEC_DIM = clip_net.config.projection_dim
print(f'CLIP loaded! Vector dim: {VEC_DIM}')
print()


def encode_queries(model):
    """Encode all query images with the given model. Returns (N, D) float32 array."""
    n_q  = len(valid_qry)
    vecs = np.zeros((n_q, VEC_DIM), dtype=np.float32)

    for s in tqdm(range(0, n_q, ENC_BATCH), desc='Encoding queries'):
        chunk = valid_qry.iloc[s : s + ENC_BATCH]
        imgs, ok = [], []
        for i, (_, row) in enumerate(chunk.iterrows()):
            try:
                imgs.append(Image.open(row['img_path']).convert('RGB'))
                ok.append(i)
            except Exception:
                pass
        if not imgs:
            continue
        inp = clip_proc(images=imgs, return_tensors='pt', padding=True).to(GPU)
        with torch.no_grad():
            raw   = model.get_image_features(**inp)
            feats = raw.pooler_output if (
                hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor)
            ) else raw
        feats = feats / feats.norm(dim=-1, keepdim=True)
        for li, gi in enumerate(ok):
            vecs[s + gi] = feats[li].cpu().numpy()
    return vecs


print('Encoding queries with frozen CLIP (for Configs A and B)...')
frozen_qry_vecs = encode_queries(clip_net)
print(f'Shape: {frozen_qry_vecs.shape}')

Loading CLIP: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded! Vector dim: 512

Encoding queries with frozen CLIP (for Configs A and B)...


Encoding queries: 100%|██████████| 223/223 [02:57<00:00,  1.26it/s]

Shape: (14218, 512)


## 5. Encode Query Images with Fine-Tuned CLIP (Config C)

In [5]:
ft_weights = os.path.join(C_INDEX_DIR, 'full_clip_finetuned.pt')

if os.path.exists(ft_weights):
    print(f'Loading fine-tuned weights: {ft_weights}')
    clip_net.load_state_dict(torch.load(ft_weights, map_location=GPU))
    clip_net.eval()
    print('Fine-tuned weights loaded ✓')

    print('Encoding queries with fine-tuned CLIP (for Config C)...')
    finetuned_qry_vecs = encode_queries(clip_net)
    print(f'Shape: {finetuned_qry_vecs.shape}')
else:
    print(f'WARNING: Fine-tuned weights not found at {ft_weights}')
    print('Config C will be skipped.')
    finetuned_qry_vecs = None

Loading fine-tuned weights: /kaggle/input/datasets/akibatra25/clip-c-output/full_clip_finetuned.pt
Fine-tuned weights loaded ✓
Encoding queries with fine-tuned CLIP (for Config C)...


Encoding queries: 100%|██████████| 223/223 [01:40<00:00,  2.21it/s]

Shape: (14218, 512)


## 6. Metric Functions

In [6]:
def ndcg_at_k(match_flags, n_relevant, k):
    """Normalised discounted cumulative gain at K."""
    dcg  = sum(1.0 / np.log2(r + 2) for r, hit in enumerate(match_flags[:k]) if hit)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(n_relevant, k)))
    return dcg / idcg if idcg > 0 else 0.0


def map_at_k(match_flags, n_relevant, k):
    """Average precision at K."""
    hits, running = 0, 0.0
    for rank, hit in enumerate(match_flags[:k], start=1):
        if hit:
            hits     += 1
            running  += hits / rank
    denom = min(n_relevant, k)
    return running / denom if denom > 0 else 0.0


print('Metric functions ready ✓')

Metric functions ready ✓


## 7. Retrieval Evaluation Function

In [7]:
def run_evaluation(faiss_path, qry_vecs, seed):
    """
    For one FAISS index and one seed:
    - Sample 20% of query images (min 500)
    - Search index → top-K hits
    - Compute Recall, NDCG, mAP at each K
    Returns a dict of metric_name -> score.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    n_samp  = min(len(valid_qry), max(500, int(0.2 * len(valid_qry))))
    s_idx   = np.random.choice(len(valid_qry), n_samp, replace=False)
    s_vecs  = qry_vecs[s_idx].astype(np.float32)
    s_ids   = valid_qry.iloc[s_idx]['item_id'].tolist()

    srch = faiss.read_index(faiss_path)
    _, hits = srch.search(s_vecs, max(K_LIST))

    per_k = {k: {'recall': [], 'ndcg': [], 'map': []} for k in K_LIST}

    for q_id, hit_row in zip(s_ids, hits):
        retrieved   = [gal_ids[i] for i in hit_row if i < len(gal_ids)]
        flags       = [1 if r == q_id else 0 for r in retrieved]
        n_rel       = gal_counts.get(q_id, 1)
        for k in K_LIST:
            per_k[k]['recall'].append(1 if any(flags[:k]) else 0)
            per_k[k]['ndcg'].append(ndcg_at_k(flags, n_rel, k))
            per_k[k]['map'].append(map_at_k(flags, n_rel, k))

    out = {}
    for k in K_LIST:
        out[f'recall@{k}'] = np.mean(per_k[k]['recall'])
        out[f'ndcg@{k}']   = np.mean(per_k[k]['ndcg'])
        out[f'map@{k}']    = np.mean(per_k[k]['map'])
    return out

## 8. Register All Configs

In [8]:
# Each entry: (display_name, faiss_filename, index_dir, query_vectors)
EVAL_CONFIGS = []

def register(name, filename, idx_dir, q_vecs):
    full = os.path.join(idx_dir, filename)
    if os.path.exists(full):
        EVAL_CONFIGS.append((name, full, q_vecs))
        print(f'  Registered: {name}')
    else:
        print(f'  SKIPPED (not found): {name} — {full}')

print('Registering configs...')
register('Config_A_beta1.0', 'idx_A_b10.bin',  AB_INDEX_DIR, frozen_qry_vecs)
register('Config_B_beta0.7', 'idx_B_b07.bin',  AB_INDEX_DIR, frozen_qry_vecs)
register('Config_B_beta0.5', 'idx_B_b05.bin',  AB_INDEX_DIR, frozen_qry_vecs)

if finetuned_qry_vecs is not None:
    register('Config_C_beta0.7', 'idx_C_b07.bin', C_INDEX_DIR, finetuned_qry_vecs)
    register('Config_C_beta0.5', 'idx_C_b05.bin', C_INDEX_DIR, finetuned_qry_vecs)

print(f'\nTotal configs to evaluate: {len(EVAL_CONFIGS)}')

Registering configs...
  Registered: Config_A_beta1.0
  Registered: Config_B_beta0.7
  Registered: Config_B_beta0.5
  Registered: Config_C_beta0.7
  Registered: Config_C_beta0.5

Total configs to evaluate: 5


## 9. Run All Evaluations

In [9]:
print('Running evaluations...\n')
aggregated = {}

for cfg_name, faiss_path, qry_vecs in EVAL_CONFIGS:
    print(f'=== {cfg_name} ===')
    seed_rows = []

    for seed in ROLL_SEEDS:
        metrics = run_evaluation(faiss_path, qry_vecs, seed)
        seed_rows.append(metrics)
        print(
            f'  Seed {seed:>4} | '
            f'R@10={metrics["recall@10"]:.4f}  '
            f'NDCG@10={metrics["ndcg@10"]:.4f}  '
            f'mAP@10={metrics["map@10"]:.4f}'
        )

    df = pd.DataFrame(seed_rows)
    aggregated[cfg_name] = {'mean': df.mean(), 'std': df.std()}

    r10_mean = df['recall@10'].mean()
    r10_std  = df['recall@10'].std()
    print(f'  MEAN  | R@10={r10_mean:.4f} ± {r10_std:.4f}')
    print()

print('All evaluations complete!')

Running evaluations...

=== Config_A_beta1.0 ===
  Seed   25 | R@10=0.5719  NDCG@10=0.2427  mAP@10=0.1692
  Seed   29 | R@10=0.5681  NDCG@10=0.2409  mAP@10=0.1680
  Seed  513 | R@10=0.5554  NDCG@10=0.2377  mAP@10=0.1671
  Seed  521 | R@10=0.5709  NDCG@10=0.2403  mAP@10=0.1657
  MEAN  | R@10=0.5666 ± 0.0076

=== Config_B_beta0.7 ===
  Seed   25 | R@10=0.5667  NDCG@10=0.2395  mAP@10=0.1676
  Seed   29 | R@10=0.5667  NDCG@10=0.2371  mAP@10=0.1651
  Seed  513 | R@10=0.5543  NDCG@10=0.2358  mAP@10=0.1660
  Seed  521 | R@10=0.5642  NDCG@10=0.2381  mAP@10=0.1650
  MEAN  | R@10=0.5630 ± 0.0059

=== Config_B_beta0.5 ===
  Seed   25 | R@10=0.5329  NDCG@10=0.2108  mAP@10=0.1442
  Seed   29 | R@10=0.5216  NDCG@10=0.2056  mAP@10=0.1399
  Seed  513 | R@10=0.5185  NDCG@10=0.2089  mAP@10=0.1439
  Seed  521 | R@10=0.5185  NDCG@10=0.2078  mAP@10=0.1408
  MEAN  | R@10=0.5229 ± 0.0068

=== Config_C_beta0.7 ===
  Seed   25 | R@10=0.9216  NDCG@10=0.6518  mAP@10=0.5577
  Seed   29 | R@10=0.9226  NDCG@10=0.64

## 10. Results Summary Table

In [10]:
print('=' * 82)
print('  EVALUATION RESULTS — Fashion Visual Product Search')
print(f'  Seeds: {ROLL_SEEDS}   |   Format: mean ± std')
print('=' * 82)
print()

for cfg_name, _, _ in EVAL_CONFIGS:
    if cfg_name not in aggregated:
        continue
    mu  = aggregated[cfg_name]['mean']
    sig = aggregated[cfg_name]['std']

    print(f'Config: {cfg_name}')
    print(f'  {"K":>4}  {"Recall@K":>20}  {"NDCG@K":>20}  {"mAP@K":>20}')
    print(f'  {"-" * 68}')
    for k in K_LIST:
        print(
            f'  K={k:>2}  '
            f'{mu[f"recall@{k}"]:.4f} ± {sig[f"recall@{k}"]:.4f}  '
            f'{mu[f"ndcg@{k}"]:.4f} ± {sig[f"ndcg@{k}"]:.4f}  '
            f'{mu[f"map@{k}"]:.4f} ± {sig[f"map@{k}"]:.4f}'
        )
    print()

  EVALUATION RESULTS — Fashion Visual Product Search
  Seeds: [25, 29, 513, 521]   |   Format: mean ± std

Config: Config_A_beta1.0
     K              Recall@K                NDCG@K                 mAP@K
  --------------------------------------------------------------------
  K= 5  0.4973 ± 0.0069  0.2369 ± 0.0032  0.1753 ± 0.0025
  K=10  0.5666 ± 0.0076  0.2404 ± 0.0021  0.1675 ± 0.0015
  K=15  0.6048 ± 0.0072  0.2470 ± 0.0022  0.1676 ± 0.0013

Config: Config_B_beta0.7
     K              Recall@K                NDCG@K                 mAP@K
  --------------------------------------------------------------------
  K= 5  0.4882 ± 0.0029  0.2321 ± 0.0023  0.1722 ± 0.0020
  K=10  0.5630 ± 0.0059  0.2376 ± 0.0016  0.1659 ± 0.0012
  K=15  0.6084 ± 0.0070  0.2453 ± 0.0018  0.1666 ± 0.0010

Config: Config_B_beta0.5
     K              Recall@K                NDCG@K                 mAP@K
  --------------------------------------------------------------------
  K= 5  0.4394 ± 0.0032  0.2004 ± 0.

## 11. Save Results and Report Best Config

In [11]:
output_rows = []
for cfg_name, _, _ in EVAL_CONFIGS:
    if cfg_name not in aggregated:
        continue
    mu  = aggregated[cfg_name]['mean']
    sig = aggregated[cfg_name]['std']
    for k in K_LIST:
        output_rows.append({
            'Config'      : cfg_name,
            'K'           : k,
            'Recall_mean' : round(mu[f'recall@{k}'], 4),
            'Recall_std'  : round(sig[f'recall@{k}'], 4),
            'NDCG_mean'   : round(mu[f'ndcg@{k}'],   4),
            'NDCG_std'    : round(sig[f'ndcg@{k}'],  4),
            'mAP_mean'    : round(mu[f'map@{k}'],    4),
            'mAP_std'     : round(sig[f'map@{k}'],   4),
        })

out_df   = pd.DataFrame(output_rows)
out_path = '/kaggle/working/evaluation_summary.csv'
out_df.to_csv(out_path, index=False)
print(f'Results written to {out_path}')

# Report best config by Recall@10
valid = [c[0] for c in EVAL_CONFIGS if c[0] in aggregated]
if valid:
    champion   = max(valid, key=lambda c: aggregated[c]['mean']['recall@10'])
    champ_r10  = aggregated[champion]['mean']['recall@10']
    print()
    print('=' * 82)
    print(f'TOP CONFIG (Recall@10): {champion}')
    print(f'Recall@10 = {champ_r10:.4f}')
    print('Use this config in the Streamlit demo!')
    print('=' * 82)

Results written to /kaggle/working/evaluation_summary.csv

TOP CONFIG (Recall@10): Config_C_beta0.7
Recall@10 = 0.9206
Use this config in the Streamlit demo!
